<a href="https://colab.research.google.com/github/nilotpalchandra/AI-assignment/blob/main/Transformers/project/transformer_attention_walkthrough.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformers from scratch: Q, K, V and attention

This notebook rebuilds the "the cat sat" example step by step with NumPy, then scales it up to a full transformer block that predicts the next word.

Run the cells from top to bottom with **Shift + Enter**. Try changing the numbers in the weight matrices and re-running to see how attention changes.

**You need:** `pip install numpy pandas`
The optional last section uses a real model (GPT-2) and also needs `pip install torch transformers`.

In [1]:
import numpy as np
import pandas as pd

np.set_printoptions(precision=3, suppress=True)

## Step 1: Turn words into numbers (embeddings)

Real models learn thousands of numbers per word. We use 2 so you can check every calculation by hand.

In [2]:
tokens = ["the", "cat", "sat"]

X = np.array([
    [1, 0],   # the
    [0, 1],   # cat
    [1, 1],   # sat
], dtype=float)

pd.DataFrame(X, index=tokens, columns=["dim 0", "dim 1"])

,dim 0,dim 1
the,1.0,0.0
cat,0.0,1.0
sat,1.0,1.0


## Step 2: The three weight matrices

In a real model these start random and are learned during training. Here they are picked by hand.
The same three matrices are used for every word.

In [3]:
W_q = np.array([[1, 0],
                [1, 1]], dtype=float)

W_k = np.array([[0, 1],
                [1, 0]], dtype=float)

W_v = np.array([[1, 2],
                [0, 1]], dtype=float)

## Step 3: Calculate Q, K and V

Each one is the embeddings multiplied by one weight matrix. In Python, `@` means matrix multiplication.

In [4]:
Q = X @ W_q
K = X @ W_k
V = X @ W_v

def show(M, name):
    return pd.DataFrame(M, index=tokens, columns=[f"{name}[{i}]" for i in range(M.shape[1])])

pd.concat([show(Q, "Q"), show(K, "K"), show(V, "V")], axis=1)

,Q[0],Q[1],K[0],K[1],V[0],V[1]
the,1.0,0.0,0.0,1.0,1.0,2.0
cat,1.0,1.0,1.0,0.0,0.0,1.0
sat,2.0,1.0,1.0,1.0,1.0,3.0


Check one by hand: "sat" = [1, 1], so its query is 1×[1, 0] + 1×[1, 1] = **[2, 1]**.

In [5]:
q_sat = Q[2]                                      # the query for "sat"
d_k = K.shape[1]                                  # size of each key = 2

scores = K @ q_sat                                # step 4: one dot product per word
scaled = scores / np.sqrt(d_k)                    # step 5
weights = np.exp(scaled) / np.exp(scaled).sum()   # step 6: softmax
output_sat = weights @ V                          # step 7

display(pd.DataFrame({
    "score (q·k)": scores,
    "scaled (÷√d_k)": scaled,
    "weight (softmax)": weights,
}, index=tokens).round(3))

print("weights add up to:", round(weights.sum(), 3))
print("new vector for 'sat':", output_sat.round(3))

,score (q·k),scaled (÷√d_k),weight (softmax)
the,1.0,0.707,0.140
cat,2.0,1.414,0.284
sat,3.0,2.121,0.576


weights add up to: 1.0
new vector for 'sat': [0.716 2.292]


## Steps 4–7: Attention for one word ("sat")

1. **Score:** dot product of sat's query with every word's key
2. **Scale:** divide by √d_k (the size of a key vector)
3. **Softmax:** turn the scores into weights that add up to 1
4. **Blend:** add up the values, each multiplied by its weight

"sat" took about 58% of its new vector from itself, 28% from "cat" and 14% from "the".
Tiny differences from the hand-worked numbers come from rounding.

## Step 8: Every word at once

The model never loops over words. One matrix multiplication gives every word's scores against every other word.

In the table below, each **row** is a word asking "who should I pay attention to?", and each **column** is a word being looked at.

In [6]:
def softmax(x):
    e = np.exp(x - x.max(axis=-1, keepdims=True))   # subtracting the max avoids overflow
    return e / e.sum(axis=-1, keepdims=True)

def attention(Q, K, V, mask=None):
    scores = Q @ K.T / np.sqrt(K.shape[-1])
    if mask is not None:
        scores = np.where(mask, scores, -np.inf)    # blocked positions get weight 0
    weights = softmax(scores)
    return weights @ V, weights

out, attn_weights = attention(Q, K, V)

print("Attention weights:")
display(pd.DataFrame(attn_weights, index=tokens, columns=tokens).round(3))
print("New vectors for every word:")
show(out, "out").round(3)

Attention weights:


,the,cat,sat
the,0.198,0.401,0.401
cat,0.248,0.248,0.503
sat,0.140,0.284,0.576


New vectors for every word:


,out[0],out[1]
the,0.599,2.000
cat,0.752,2.255
sat,0.716,2.292


The "sat" row matches Step 7.

## Causal mask (how GPT-style models work)

When generating text, a word must not see the words that come after it.
Future positions are blocked by setting their scores to −∞ before softmax, which makes their weight 0.

In [7]:
causal = np.tri(len(tokens), dtype=bool)   # True = allowed to look
print("Allowed to look:")
display(pd.DataFrame(causal, index=tokens, columns=tokens))

out_masked, masked_weights = attention(Q, K, V, mask=causal)
print("Attention weights with the mask:")
pd.DataFrame(masked_weights, index=tokens, columns=tokens).round(3)

Allowed to look:


,the,cat,sat
the,True,False,False
cat,True,True,False
sat,True,True,True


Attention weights with the mask:


,the,cat,sat
the,1.00,0.000,0.000
cat,0.50,0.500,0.000
sat,0.14,0.284,0.576


"the" can only see itself, "cat" sees "the" and itself, and "sat" sees everything, so its row is unchanged.

## Why divide by √d_k?

With real-sized vectors, dot products get large. Without scaling, softmax puts almost all the weight on a single word, and training struggles.
This compares the largest weight with and without scaling for random 512-number vectors:

In [8]:
rng = np.random.default_rng(0)
d = 512
q = rng.normal(size=d)
keys = rng.normal(size=(10, d))     # 10 words

raw = keys @ q
print("largest weight WITHOUT scaling:", softmax(raw).max().round(3))
print("largest weight WITH scaling   :", softmax(raw / np.sqrt(d)).max().round(3))

largest weight WITHOUT scaling: 1.0
largest weight WITH scaling   : 0.426


## Multiple heads

Real models run several attention calculations ("heads") side by side, each with its own W_q, W_k and W_v.
Each head works on smaller vectors (model size ÷ number of heads). Their outputs are joined back together and mixed by one more matrix, W_o.

Here: 5 words, 8 numbers per word, 2 heads with 4 numbers each. The weights are random, as they would be before training.

In [9]:
rng = np.random.default_rng(42)

d_model, n_heads = 8, 2
d_head = d_model // n_heads
words = ["the", "animal", "was", "very", "tired"]

E = rng.normal(size=(len(words), d_model))       # embeddings

def init_head():
    return {name: rng.normal(size=(d_model, d_head)) * 0.5 for name in ("W_q", "W_k", "W_v")}

def multi_head_attention(x, heads, W_o, mask=None):
    outputs, all_weights = [], []
    for h in heads:
        out, w = attention(x @ h["W_q"], x @ h["W_k"], x @ h["W_v"], mask)
        outputs.append(out)
        all_weights.append(w)
    return np.concatenate(outputs, axis=-1) @ W_o, all_weights

heads = [init_head() for _ in range(n_heads)]
W_o = rng.normal(size=(d_model, d_model)) * 0.5
mask = np.tri(len(words), dtype=bool)

mha_out, head_weights = multi_head_attention(E, heads, W_o, mask)

print("input shape :", E.shape)
print("output shape:", mha_out.shape, "(same as the input, so blocks can be stacked)")
for i, w in enumerate(head_weights):
    print(f"\nHead {i} attention weights:")
    display(pd.DataFrame(w, index=words, columns=words).round(2))

input shape : (5, 8)
output shape: (5, 8) (same as the input, so blocks can be stacked)

Head 0 attention weights:


,the,animal,was,very,tired
the,1.00,0.00,0.00,0.00,0.00
animal,0.71,0.29,0.00,0.00,0.00
was,0.27,0.47,0.27,0.00,0.00
very,0.29,0.28,0.23,0.20,0.00
tired,0.09,0.38,0.13,0.35,0.05



Head 1 attention weights:


,the,animal,was,very,tired
the,1.00,0.00,0.00,0.00,0.00
animal,0.01,0.99,0.00,0.00,0.00
was,0.05,0.89,0.06,0.00,0.00
very,0.01,0.48,0.11,0.40,0.00
tired,0.22,0.10,0.23,0.16,0.29


The two heads give different weights on the same sentence because each has its own matrices.
With random weights the patterns mean nothing yet; training is what makes them useful.

## A full transformer block

Attention lets words share information. The feed-forward network (FFN) then processes each word on its own.
Residual connections (the `x + ...` part) and layer normalization keep the numbers stable.

```
x = x + Attention(LayerNorm(x))
x = x + FFN(LayerNorm(x))
```

In [10]:
def layer_norm(x, eps=1e-5):
    mean = x.mean(axis=-1, keepdims=True)
    std = x.std(axis=-1, keepdims=True)
    return (x - mean) / (std + eps)

def ffn(x, W1, W2):
    return np.maximum(0, x @ W1) @ W2      # expand, ReLU, shrink back

def init_block():
    return {
        "heads": [init_head() for _ in range(n_heads)],
        "W_o": rng.normal(size=(d_model, d_model)) * 0.5,
        "W1": rng.normal(size=(d_model, 4 * d_model)) * 0.3,   # 8 -> 32
        "W2": rng.normal(size=(4 * d_model, d_model)) * 0.3,   # 32 -> 8
    }

def transformer_block(x, block, mask):
    attn_out, _ = multi_head_attention(layer_norm(x), block["heads"], block["W_o"], mask)
    x = x + attn_out
    x = x + ffn(layer_norm(x), block["W1"], block["W2"])
    return x

blocks = [init_block() for _ in range(3)]   # a tiny 3-layer model

x = E
for i, block in enumerate(blocks):
    x = transformer_block(x, block, mask)
    print(f"after layer {i + 1}: shape {x.shape}")

after layer 1: shape (5, 8)
after layer 2: shape (5, 8)
after layer 3: shape (5, 8)


## Final step: predict the next word

The last word's vector is multiplied by one more matrix to get a score for every word in the vocabulary, and softmax turns those scores into probabilities.

This model is untrained, so its guess is random. Training adjusts every weight in this notebook until these probabilities become good.

In [11]:
vocab = ["the", "animal", "was", "very", "tired", "slept", "ran", "."]
W_out = rng.normal(size=(d_model, len(vocab)))

logits = layer_norm(x)[-1] @ W_out      # use the last word's vector
probs = softmax(logits)

pd.Series(probs, index=vocab, name="probability").sort_values(ascending=False).round(3)

,probability
the,0.531
tired,0.209
was,0.103
slept,0.103
animal,0.039
.,0.009
ran,0.005
very,0.001


## Recap

| Step | Code |
|---|---|
| Q, K, V | `X @ W_q`, `X @ W_k`, `X @ W_v` |
| Scores | `Q @ K.T / sqrt(d_k)` |
| Weights | `softmax(scores)` |
| Output | `weights @ V` |

Everything else (heads, the FFN, layers, prediction) is built around those four lines.

## Optional: attention inside a real model (GPT-2)

This downloads GPT-2 (about 500 MB) the first time, so it needs internet and `pip install torch transformers`.
It prints which words the token "it" pays the most attention to in each of GPT-2's 12 layers, averaged over the heads.

The very first token is skipped, because real models often use it as a default "parking spot" for attention they don't need.

In [12]:
try:
    import torch
    from transformers import AutoTokenizer, AutoModel
except ImportError:
    print("Skipped: run  pip install torch transformers  and then re-run this cell.")
else:
    tok = AutoTokenizer.from_pretrained("gpt2")
    model = AutoModel.from_pretrained("gpt2", attn_implementation="eager")

    sentence = "The animal didn't cross the street because it was too tired"
    inputs = tok(sentence, return_tensors="pt")
    with torch.no_grad():
        attentions = model(**inputs, output_attentions=True).attentions   # one per layer

    toks = [t.replace("Ġ", "") for t in tok.convert_ids_to_tokens(inputs["input_ids"][0])]
    it_pos = toks.index("it")

    for layer, att in enumerate(attentions):
        avg = att[0, :, it_pos, :].mean(dim=0)    # average over heads
        top = avg[1:].topk(3)                     # skip the first token
        picks = ", ".join(f"{toks[i + 1]} ({v:.2f})"
                          for v, i in zip(top.values.tolist(), top.indices.tolist()))
        print(f"layer {layer:2d}: {picks}")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

layer  0: it (0.35), because (0.13), the (0.08)
layer  1: it (0.18), because (0.14), the (0.09)
layer  2: because (0.31), it (0.11), street (0.08)
layer  3: because (0.28), it (0.09), animal (0.04)
layer  4: because (0.19), animal (0.09), it (0.08)
layer  5: cross (0.17), because (0.10), animal (0.06)
layer  6: because (0.12), animal (0.06), cross (0.06)
layer  7: because (0.12), cross (0.04), street (0.04)
layer  8: because (0.13), street (0.04), animal (0.03)
layer  9: because (0.08), animal (0.06), it (0.04)
layer 10: didn (0.06), because (0.04), cross (0.03)
layer 11: 't (0.08), it (0.06), animal (0.05)
